# Task 6: Feature Selection and Recommendation Quality Evaluation

## Objective

The objective of this task is to experiment with different combinations of text features and identify which combination produces more relevant TED Talk recommendations.

## Description

In this task, we compare four feature combinations using TF-IDF vectorization and cosine similarity.

The experiments use the following combinations:

1. Title only
2. Title + Tags
3. Title + Description + Tags
4. Title + Description + Transcript + Tags

For each experiment, we generate recommendations for at least five selected TED Talks and compare the results.

We manually examine the recommendations to identify which feature combination provides more meaningful and relevant results.

## Steps Performed

1. Loaded the cleaned TED Talks dataset.
2. Prepared the text columns.
3. Selected five TED Talks for testing.
4. Generated TF-IDF vectors for each feature combination.
5. Calculated cosine similarity for each experiment.
6. Generated the top five recommendations for each selected talk.
7. Compared the recommendation results.
8. Evaluated the relevance of the recommendations.
9. Selected the best feature combination based on the observations.

## Conclusion

This task helps us understand how different text features affect recommendation quality and helps us select suitable features for the TED Talks recommendation system.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from IPython.display import display

In [2]:
df = pd.read_csv(r"C:\Users\vramy\OneDrive\Desktop\samba-shiva-reddy\ted-talks-recommender\data\processed\ted_talks_cleaned.csv")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())

Dataset shape: (2550, 9)
Columns: ['title', 'description', 'tags', 'main_speaker', 'speaker_occupation', 'event', 'views', 'url', 'transcript']


,title,description,tags,main_speaker,speaker_occupation,event,views,url,transcript
0,do schools kill creativity?,sir ken robinson makes an entertaining and pro...,children creativity culture dance education pa...,Ken Robinson,Author/educator,TED2006,47227110,https://www.ted.com/talks/ken_robinson_says_sc...,"good morning. how are you? it's been great, ha..."
1,averting the climate crisis,with the same humor and humanity he exuded in ...,alternative energy cars climate change culture...,Al Gore,Climate advocate,TED2006,3200520,https://www.ted.com/talks/al_gore_on_averting_...,"thank you so much, chris. and it's truly a gre..."
2,simplicity sells,new york times columnist david pogue takes aim...,computers entertainment interface design media...,David Pogue,Technology columnist,TED2006,1636292,https://www.ted.com/talks/david_pogue_says_sim...,"hello voice mail, my old friend. i've called f..."
3,greening the ghetto,"in an emotionally charged talk, macarthur-winn...",macarthur grant activism business cities envir...,Majora Carter,Activist for environmental justice,TED2006,1697550,https://www.ted.com/talks/majora_carter_s_tale...,if you're here today — and i'm very happy that...
4,the best stats you've ever seen,you've never seen data presented like this. wi...,africa asia google demo economics global devel...,Hans Rosling,Global health expert; data visionary,TED2006,12005869,https://www.ted.com/talks/hans_rosling_shows_t...,"about 10 years ago, i took on the task to teac..."


In [3]:
text_columns = ["title", "tags", "description", "transcript"]

for col in text_columns:
    df[col] = df[col].fillna("").astype(str)

# Remove records without a title
talks = df[df["title"].str.strip().ne("")].copy()

talks.reset_index(drop=True, inplace=True)

print("Total TED Talks available:", len(talks))

display(talks[text_columns].head())

Total TED Talks available: 2550


,title,tags,description,transcript
0,do schools kill creativity?,children creativity culture dance education pa...,sir ken robinson makes an entertaining and pro...,"good morning. how are you? it's been great, ha..."
1,averting the climate crisis,alternative energy cars climate change culture...,with the same humor and humanity he exuded in ...,"thank you so much, chris. and it's truly a gre..."
2,simplicity sells,computers entertainment interface design media...,new york times columnist david pogue takes aim...,"hello voice mail, my old friend. i've called f..."
3,greening the ghetto,macarthur grant activism business cities envir...,"in an emotionally charged talk, macarthur-winn...",if you're here today — and i'm very happy that...
4,the best stats you've ever seen,africa asia google demo economics global devel...,you've never seen data presented like this. wi...,"about 10 years ago, i took on the task to teac..."


In [4]:
# Select five talks with different titles
candidate_talks = talks.drop_duplicates(subset=["title"])

selected_talks = candidate_talks.sample(
    n=5,
    random_state=42
)

selected_indices = selected_talks.index.tolist()

print("Selected TED Talks:")

display(
    selected_talks[["title", "main_speaker"]].reset_index(drop=True)
)

Selected TED Talks:


,title,main_speaker
0,"how the ""ghost map"" helped end a killer disease",Steven Johnson
1,juggle and jest,Raspyni Brothers
2,3 reasons why we can win the fight against pov...,Andrew Youn
3,a hospital tour in nigeria,Seyi Oyesola
4,the danger of hiding who you are,Morgana Bailey


In [5]:
experiments = {
    "Experiment 1 - Title Only": ["title"],

    "Experiment 2 - Title + Tags": [
        "title", "tags"
    ],

    "Experiment 3 - Title + Description + Tags": [
        "title", "description", "tags"
    ],

    "Experiment 4 - Title + Description + Transcript + Tags": [
        "title", "description", "transcript", "tags"
    ]
}

for experiment, features in experiments.items():
    print(experiment, ":", features)

Experiment 1 - Title Only : ['title']
Experiment 2 - Title + Tags : ['title', 'tags']
Experiment 3 - Title + Description + Tags : ['title', 'description', 'tags']
Experiment 4 - Title + Description + Transcript + Tags : ['title', 'description', 'transcript', 'tags']


In [6]:
def generate_experiment_recommendations(
    talks,
    selected_indices,
    features,
    experiment_name,
    top_n=5
):
    # Combine the selected features into one text column
    combined_text = talks[features].agg(" ".join, axis=1)

    # Generate TF-IDF vectors
    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=10000,
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b"
    )

    tfidf_matrix = vectorizer.fit_transform(combined_text)

    # Calculate cosine similarity once for this experiment
    similarity_matrix = cosine_similarity(tfidf_matrix)

    print(f"\n{experiment_name}")
    print("TF-IDF matrix shape:", tfidf_matrix.shape)
    print("Similarity matrix shape:", similarity_matrix.shape)

    experiment_results = []

    # Generate recommendations for all five selected talks
    for talk_index in selected_indices:

        scores = similarity_matrix[talk_index].copy()

        # Exclude the selected talk itself
        scores[talk_index] = -1

        # Sort scores in descending order
        ranked_indices = np.argsort(scores)[::-1]

        # Keep only talks with positive similarity
        ranked_indices = [
            idx for idx in ranked_indices
            if scores[idx] > 0
        ][:top_n]

        # Create recommendation DataFrame
        recommendations = talks.iloc[ranked_indices][
            ["title", "main_speaker", "url"]
        ].copy()

        recommendations["similarity_score"] = [
            scores[idx] for idx in ranked_indices
        ]

        recommendations.insert(
            0,
            "selected_talk",
            talks.loc[talk_index, "title"]
        )

        recommendations.insert(
            0,
            "experiment",
            experiment_name
        )

        experiment_results.append(recommendations)

    return pd.concat(experiment_results, ignore_index=True)

In [7]:
all_experiment_results = []

for experiment_name, features in experiments.items():

    results = generate_experiment_recommendations(
        talks=talks,
        selected_indices=selected_indices,
        features=features,
        experiment_name=experiment_name,
        top_n=5
    )

    all_experiment_results.append(results)

# Combine results from all four experiments
final_results = pd.concat(
    all_experiment_results,
    ignore_index=True
)

print("All experiment results:")
display(final_results)


Experiment 1 - Title Only
TF-IDF matrix shape: (2550, 3408)
Similarity matrix shape: (2550, 2550)

Experiment 2 - Title + Tags
TF-IDF matrix shape: (2550, 3503)
Similarity matrix shape: (2550, 2550)

Experiment 3 - Title + Description + Tags
TF-IDF matrix shape: (2550, 10000)
Similarity matrix shape: (2550, 2550)

Experiment 4 - Title + Description + Transcript + Tags
TF-IDF matrix shape: (2550, 10000)
Similarity matrix shape: (2550, 2550)
All experiment results:


,experiment,selected_talk,title,main_speaker,url,similarity_score
0,Experiment 1 - Title Only,"how the ""ghost map"" helped end a killer disease",a map of the brain,Allan Jones,https://www.ted.com/talks/allan_jones_a_map_of...,0.351115
1,Experiment 1 - Title Only,"how the ""ghost map"" helped end a killer disease",how megacities are changing the map of the world,Parag Khanna,https://www.ted.com/talks/parag_khanna_how_meg...,0.236216
2,Experiment 1 - Title Only,"how the ""ghost map"" helped end a killer disease",exploring the mind of a killer,Jim Fallon,https://www.ted.com/talks/jim_fallon_exploring...,0.229619
3,Experiment 1 - Title Only,"how the ""ghost map"" helped end a killer disease",how architecture helped music evolve,David Byrne,https://www.ted.com/talks/david_byrne_how_arch...,0.215420
4,Experiment 1 - Title Only,"how the ""ghost map"" helped end a killer disease",10 ways the world could end,Stephen Petranek,https://www.ted.com/talks/stephen_petranek_cou...,0.213550
...,...,...,...,...,...,...
90,Experiment 4 - Title + Description + Transcrip...,the danger of hiding who you are,the myth of the gay agenda,LZ Granderson,https://www.ted.com/talks/lz_granderson_the_my...,0.376174
91,Experiment 4 - Title + Description + Transcrip...,the danger of hiding who you are,what the gay rights movement learned from the ...,Yoruba Richen,https://www.ted.com/talks/yoruba_richen_what_t...,0.256726
92,Experiment 4 - Title + Description + Transcrip...,the danger of hiding who you are,this is what lgbt life is like around the world,Jenni Chang and Lisa Dazols,https://www.ted.com/talks/jenni_chang_and_lisa...,0.243134
93,Experiment 4 - Title + Description + Transcrip...,the danger of hiding who you are,a message to gay teens: it gets better,Joel Burns,https://www.ted.com/talks/joel_burns_tells_gay...,0.226531


In [8]:
for selected_title in selected_talks["title"]:

    print("\nSelected TED Talk:")
    print(selected_title)

    selected_results = final_results[
        final_results["selected_talk"] == selected_title
    ]

    for experiment_name in experiments.keys():

        print("\n", experiment_name)

        experiment_output = selected_results[
            selected_results["experiment"] == experiment_name
        ][["title", "similarity_score"]]

        display(experiment_output.reset_index(drop=True))


Selected TED Talk:
how the "ghost map" helped end a killer disease

 Experiment 1 - Title Only


,title,similarity_score
0,a map of the brain,0.351115
1,how megacities are changing the map of the world,0.236216
2,exploring the mind of a killer,0.229619
3,how architecture helped music evolve,0.215420
4,10 ways the world could end,0.213550



 Experiment 2 - Title + Tags


,title,similarity_score
0,happy maps,0.304960
1,how megacities are changing the map of the world,0.293069
2,a next-gen cure for killer infections,0.254884
3,a map of the brain,0.254658
4,the killer american diet that's sweeping the p...,0.251030



 Experiment 3 - Title + Description + Tags


,title,similarity_score
0,"making maps to fight disaster, build economies",0.266598
1,how a video game might help us build better ci...,0.222416
2,maps that show us who we are,0.219214
3,7 principles for building better cities,0.193200
4,a map of the brain,0.187594


... (30 outputs left out)


 Experiment 3 - Title + Description + Tags


,title,similarity_score
0,the laws that sex workers really want,0.122466
1,we're all hiding something. let's find the cou...,0.121771
2,this is what lgbt life is like around the world,0.119902
3,"to raise brave girls, encourage adventure",0.116685
4,what will you tell your daughters about 2016?,0.115045



 Experiment 4 - Title + Description + Transcript + Tags


,title,similarity_score
0,the myth of the gay agenda,0.376174
1,what the gay rights movement learned from the ...,0.256726
2,this is what lgbt life is like around the world,0.243134
3,a message to gay teens: it gets better,0.226531
4,fifty shades of gay,0.225443


In [9]:
comparison_table = final_results.pivot_table(
    index=["selected_talk", "experiment"],
    values="title",
    aggfunc=lambda x: list(x)
).reset_index()

comparison_table.rename(
    columns={"title": "recommended_talks"},
    inplace=True
)

display(comparison_table)

,selected_talk,experiment,recommended_talks
0,3 reasons why we can win the fight against pov...,Experiment 1 - Title Only,"[social experiments to fight poverty, how mobi..."
1,3 reasons why we can win the fight against pov...,Experiment 2 - Title + Tags,"[the ""bottom billion"", the good news on povert..."
2,3 reasons why we can win the fight against pov...,Experiment 3 - Title + Description + Tags,"[the story we tell about poverty isn't true, t..."
3,3 reasons why we can win the fight against pov...,Experiment 4 - Title + Description + Transcrip...,[poverty isn't a lack of character; it's a lac...
4,a hospital tour in nigeria,Experiment 1 - Title Only,"[a tour of nollywood, nigeria's booming film i..."
5,a hospital tour in nigeria,Experiment 2 - Title + Tags,"[crowdsource your health, world-class health c..."
6,a hospital tour in nigeria,Experiment 3 - Title + Description + Tags,"[take health care off the mainframe, a tempora..."
7,a hospital tour in nigeria,Experiment 4 - Title + Description + Transcrip...,"[world-class health care, the universal anesth..."
8,"how the ""ghost map"" helped end a killer disease",Experiment 1 - Title Only,"[a map of the brain, how megacities are changi..."
9,"how the ""ghost map"" helped end a killer disease",Experiment 2 - Title + Tags,"[happy maps, how megacities are changing the m..."


In [21]:
evaluation_results = final_results.copy()

relevance_values = [
    # Rows 0-4
    0, 0, 0, 0, 0,

    # Rows 5-9
    1, 1, 0, 1, 0,

    # Rows 10-14
    0, 0, 0, 0, 1,

    # Rows 15-19
    1, 1, 1, 0, 0,

    # Rows 20-24
    0, 0, 1, 0, 0,

    # Rows 25-29
    0, 0, 0, 0, 0,

    # Rows 30-34
    1, 1, 1, 1, 1,

    # Rows 35-39
    1, 1, 1, 1, 1,

    # Rows 40-44
    1, 1, 1, 1, 1,

    # Rows 45-49
    1, 0, 1, 0, 1,

    # Rows 50-54
    0, 0, 1, 0, 0,

    # Rows 55-59
    1, 1, 1, 1, 1,

    # Rows 60-64
    1, 1, 1, 1, 1,

    # Rows 65-69
    0, 1, 1, 0, 0,

    # Rows 70-74
    1, 1, 0, 1, 0,

    # Rows 75-79
    1, 0, 0, 1, 1,

    # Rows 80-84
    1, 1, 1, 1, 1,

    # Rows 85-89
    1, 1, 1, 0, 0,

    # Rows 90-94
    1, 1, 1, 1, 1
]

# Check the number of relevance values
print("Number of recommendation rows:", len(final_results))
print("Number of relevance values:", len(relevance_values))

assert len(final_results) == len(relevance_values)

# Assign relevance values
evaluation_results["relevant"] = relevance_values

# Display evaluation results
display(
    evaluation_results[
        ["experiment", "selected_talk", "title",
         "similarity_score", "relevant"]
    ]
)

Number of recommendation rows: 95
Number of relevance values: 95


,experiment,selected_talk,title,similarity_score,relevant
0,Experiment 1 - Title Only,"how the ""ghost map"" helped end a killer disease",a map of the brain,0.351115,0
1,Experiment 1 - Title Only,"how the ""ghost map"" helped end a killer disease",how megacities are changing the map of the world,0.236216,0
2,Experiment 1 - Title Only,"how the ""ghost map"" helped end a killer disease",exploring the mind of a killer,0.229619,0
3,Experiment 1 - Title Only,"how the ""ghost map"" helped end a killer disease",how architecture helped music evolve,0.215420,0
4,Experiment 1 - Title Only,"how the ""ghost map"" helped end a killer disease",10 ways the world could end,0.213550,0
...,...,...,...,...,...
90,Experiment 4 - Title + Description + Transcrip...,the danger of hiding who you are,the myth of the gay agenda,0.376174,1
91,Experiment 4 - Title + Description + Transcrip...,the danger of hiding who you are,what the gay rights movement learned from the ...,0.256726,1
92,Experiment 4 - Title + Description + Transcrip...,the danger of hiding who you are,this is what lgbt life is like around the world,0.243134,1
93,Experiment 4 - Title + Description + Transcrip...,the danger of hiding who you are,a message to gay teens: it gets better,0.226531,1


In [22]:
evaluation_results["relevant"] = pd.to_numeric(
    evaluation_results["relevant"],
    errors="coerce"
)

if evaluation_results["relevant"].isna().any():
    print("Please complete all relevance judgments using 1 or 0.")

else:
    precision_results = (
        evaluation_results
        .groupby(["experiment", "selected_talk"])
        .agg(
            relevant_count=("relevant", "sum"),
            recommendation_count=("relevant", "count")
        )
        .reset_index()
    )

    precision_results["precision"] = (
        precision_results["relevant_count"] /
        precision_results["recommendation_count"]
    )

    display(precision_results)

    experiment_precision = (
        precision_results
        .groupby("experiment")["precision"]
        .mean()
        .reset_index(name="mean_precision")
        .sort_values("mean_precision", ascending=False)
    )

    print("Average Precision for each experiment:")
    display(experiment_precision)

,experiment,selected_talk,relevant_count,recommendation_count,precision
0,Experiment 1 - Title Only,3 reasons why we can win the fight against pov...,3,5,0.6
1,Experiment 1 - Title Only,a hospital tour in nigeria,1,5,0.2
2,Experiment 1 - Title Only,"how the ""ghost map"" helped end a killer disease",0,5,0.0
3,Experiment 1 - Title Only,the danger of hiding who you are,3,5,0.6
4,Experiment 2 - Title + Tags,3 reasons why we can win the fight against pov...,5,5,1.0
5,Experiment 2 - Title + Tags,a hospital tour in nigeria,5,5,1.0
6,Experiment 2 - Title + Tags,"how the ""ghost map"" helped end a killer disease",1,5,0.2
7,Experiment 2 - Title + Tags,juggle and jest,0,5,0.0
8,Experiment 2 - Title + Tags,the danger of hiding who you are,5,5,1.0
9,Experiment 3 - Title + Description + Tags,3 reasons why we can win the fight against pov...,5,5,1.0


Average Precision for each experiment:


,experiment,mean_precision
3,Experiment 4 - Title + Description + Transcrip...,0.76
1,Experiment 2 - Title + Tags,0.64
2,Experiment 3 - Title + Description + Tags,0.64
0,Experiment 1 - Title Only,0.35


# Observations

1. In Experiment 1, only the TED Talk title was used for generating recommendations.
2. In Experiment 2, tags were added along with titles to provide additional topic information.
3. In Experiment 3, descriptions were also included to provide more information about each TED Talk.
4. In Experiment 4, transcripts were included along with titles, descriptions, and tags.
5. The recommendations changed when different combinations of features were used.
6. Adding more textual features can provide additional information, but it does not always guarantee better recommendations.
7. Precision@5 helps compare the relevance of recommendations across the four experiments.
8. Manual inspection is important because a higher similarity score does not always mean that a recommendation is relevant.

## Limitations

1. Recommendation quality depends on the quality of the available text data.
2. TF-IDF focuses on word importance and does not fully understand the meaning of sentences.
3. Long transcripts may contain many words that are not directly related to the main topic.
4. Manual relevance evaluation may differ from one person to another.
5. The experiments use only five selected TED Talks, so the results may not represent the entire dataset.

## Conclusion

The experiments help us understand how different text features affect TED Talk recommendations. By comparing recommendation results and Precision@5, we can identify which feature combination provides more relevant recommendations for our selected talks.


# Final Feature Selection

## Selected Feature Combination

The feature combination selected for the TED Talks recommendation system is:

**Experiment 4: Title + Description + Transcript + Tags**

## Justification

After comparing all four experiments, I selected this feature combination because:

1. It produced more relevant recommendations for the selected TED Talks.
2. It provided useful textual information for comparing different talks.
3. The recommendations were more meaningful based on manual inspection.
4. Its average Precision@5 was **0.76**.

## Final Conclusion

Based on the recommendation results and manual evaluation, **Experiment 4 (Title + Description + Transcript + Tags)** performed better among the four tested combinations.

This feature combination will be considered for the final TED Talks recommendation system.
